<a href="https://colab.research.google.com/github/etmcrae/Who-Got-DOGEd-/blob/main/DOGE_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
** Department of Government Efficiency (DOGE) API ** **bold text**
```



**Contains Savings & Payments**

Savings include: grants, contracts, leases

Payments include: payments, statistics

Source is https://doge.gov/savings

**From https://github.com/m-nolan/doge-scrape/blob/main/doge-scrape.py**

might need to update the USAspending data from https://api.usaspending.gov/docs/endpoints

In [ ]:
import os
from datetime import datetime
from time import sleep

import numpy as np
import pandas as pd
import requests as req
!pip install validators
!pip install ratelimit # Install the ratelimit package
from bs4 import BeautifulSoup
from ratelimit import limits, sleep_and_retry
!pip install selenium
from selenium.webdriver import Firefox
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from tqdm import tqdm

In [ ]:
!wget https://files.usaspending.gov/award_data_archive/FY2025_All_Contracts_Full_20250506.zip
!unzip FY2025_All_Contracts_Full_20250506.zip

--2025-06-06 19:24:50--  https://files.usaspending.gov/award_data_archive/FY2025_All_Contracts_Full_20250506.zip
Resolving files.usaspending.gov (files.usaspending.gov)... 166.123.8.118, 2610:108:4100:100c::9:168
Connecting to files.usaspending.gov (files.usaspending.gov)|166.123.8.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 739753150 (705M) [application/zip]
Saving to: ‘FY2025_All_Contracts_Full_20250506.zip’

FY2025_All_Contract 100%[===================>] 705.48M  48.6MB/s    in 18s     

2025-06-06 19:25:09 (39.1 MB/s) - ‘FY2025_All_Contracts_Full_20250506.zip’ saved [739753150/739753150]

Archive:  FY2025_All_Contracts_Full_20250506.zip
  inflating: FY2025_All_Contracts_Full_20250508_1.csv  
  inflating: FY2025_All_Contracts_Full_20250508_2.csv  
  inflating: FY2025_All_Contracts_Full_20250508_3.csv  


In [ ]:
!ls -la


total 5916284
drwxr-xr-x 1 root root       4096 Jun  6 19:25 .
drwxr-xr-x 1 root root       4096 Jun  6 19:23 ..
drwxr-xr-x 4 root root       4096 Jun  5 13:38 .config
-rw-r--r-- 1 root root  739753150 May 13 19:08 FY2025_All_Contracts_Full_20250506.zip
-rw-r--r-- 1 root root 2179860288 May  9 01:52 FY2025_All_Contracts_Full_20250508_1.csv
-rw-r--r-- 1 root root 2099189096 May  9 01:53 FY2025_All_Contracts_Full_20250508_2.csv
-rw-r--r-- 1 root root 1039433226 May  9 01:54 FY2025_All_Contracts_Full_20250508_3.csv
drwxr-xr-x 1 root root       4096 Jun  5 13:38 sample_data


In [ ]:
!unzip -l ../FY2025_All_Contracts_Full_20250506.zip
!unzip ../FY2025_All_Contracts_Full_20250506.zip


unzip:  cannot find or open ../FY2025_All_Contracts_Full_20250506.zip, ../FY2025_All_Contracts_Full_20250506.zip.zip or ../FY2025_All_Contracts_Full_20250506.zip.ZIP.
unzip:  cannot find or open ../FY2025_All_Contracts_Full_20250506.zip, ../FY2025_All_Contracts_Full_20250506.zip.zip or ../FY2025_All_Contracts_Full_20250506.zip.ZIP.
total 5916284
drwxr-xr-x 1 root root       4096 Jun  6 19:25 .
drwxr-xr-x 1 root root       4096 Jun  6 19:23 ..
drwxr-xr-x 4 root root       4096 Jun  5 13:38 .config
-rw-r--r-- 1 root root  739753150 May 13 19:08 FY2025_All_Contracts_Full_20250506.zip
-rw-r--r-- 1 root root 2179860288 May  9 01:52 FY2025_All_Contracts_Full_20250508_1.csv
-rw-r--r-- 1 root root 2099189096 May  9 01:53 FY2025_All_Contracts_Full_20250508_2.csv
-rw-r--r-- 1 root root 1039433226 May  9 01:54 FY2025_All_Contracts_Full_20250508_3.csv
drwxr-xr-x 1 root root       4096 Jun  5 13:38 sample_data


<ipython-input-7-5cbc6ca64e50>:4: DtypeWarning: Columns (5,47,68,71,72,81,107,108,115,116,117,118,135,136,164,165,171,172,179,180) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv')


In [ ]:
!ls -la


total 5916284
drwxr-xr-x 1 root root       4096 Jun  6 19:25 .
drwxr-xr-x 1 root root       4096 Jun  6 19:23 ..
drwxr-xr-x 4 root root       4096 Jun  5 13:38 .config
-rw-r--r-- 1 root root  739753150 May 13 19:08 FY2025_All_Contracts_Full_20250506.zip
-rw-r--r-- 1 root root 2179860288 May  9 01:52 FY2025_All_Contracts_Full_20250508_1.csv
-rw-r--r-- 1 root root 2099189096 May  9 01:53 FY2025_All_Contracts_Full_20250508_2.csv
-rw-r--r-- 1 root root 1039433226 May  9 01:54 FY2025_All_Contracts_Full_20250508_3.csv
drwxr-xr-x 1 root root       4096 Jun  5 13:38 sample_data


In [ ]:
!wc FY2025_All_Contracts_Full_20250508_*

   1000001  123129882 2179860288 FY2025_All_Contracts_Full_20250508_1.csv
   1000001  113663795 2099189096 FY2025_All_Contracts_Full_20250508_2.csv
    500727   55654447 1039433226 FY2025_All_Contracts_Full_20250508_3.csv
   2500729  292448124 5318482610 total


In [ ]:
df0 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv', nrows=3)


In [ ]:
cols3 = list(df0.columns)

In [ ]:
len(cols3), len(cols2), len(cols1)

(297, 297, 297)

In [ ]:
cols1==cols2==cols3

True

In [ ]:
df1 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', nrows=100)

In [ ]:
# keep = [
    2,
    10,
    15,
    23,
    26,
    29,
    31,
    35,
    37,
    44,
    50,
    53,
    59,
    60,
    62,
    64,
    65,
    66,
    67,
    68.
    71,
    109,
    110,
    144,
    159,
    161,
    163,
    176,
    187,
    186,
    187,
    194,
    195:296
   ]

keep

In [ ]:
df1.describe(include='all').transpose().reset_index()

,index,count,unique,top,freq,mean,std,min,25%,50%,75%,max
0,contract_transaction_unique_key,100,100,4732_-NONE-_47QSWA20D0092_PA0042_-NONE-_-NONE-,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,contract_award_unique_key,100,100,CONT_IDV_47QSWA20D0092_4732,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,award_id_piid,100,100,47QSWA20D0092,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,modification_number,100,17,PSA897,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,transaction_number,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,parent_award_agency_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,parent_award_agency_name,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,parent_award_id_piid,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,parent_award_modification_number,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,federal_action_obligation,100.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:

keep_cols_indices = [
    2, 10, 15, 23, 26, 29, 31, 35, 37, 44, 50, 53, 59, 60, 62, 64, 65, 66, 67, 68, 71,
    109, 110, 144, 159, 161, 163, 176, 187, 186, 187, 194, 195
]

# Add the range from 195 to 296 (inclusive)
keep_cols_indices.extend(range(196, 297))

# Get the column names from the first few rows
temp_df = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', nrows=100)
all_cols = list(temp_df.columns)

# Select the column names based on the indices
keep_cols_names = [all_cols[i] for i in keep_cols_indices if i < len(all_cols)]

# Read the full CSV file, keeping only the specified columns
df_subset = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', usecols=keep_cols_names)

# Display the first few rows of the new dataframe
print(df_subset.head())

# Display info about the new dataframe
df_subset.info()


<ipython-input-29-bbd2867d413a>:17: DtypeWarning: Columns (68,71) have mixed types. Specify dtype option on import or set low_memory=False.
  df_subset = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', usecols=keep_cols_names)


   award_id_piid  total_dollars_obligated  potential_total_value_of_award  \
0  47QSWA20D0092                      0.0                        500000.0   
1  47QRAA24D006P                      0.0                        500000.0   
2  47QTCA20D008C                      0.0                       4000000.0   
3  47QTCA23D0009                      0.0                       7600000.0   
4     GS33F004DA                      0.0                        450000.0   

  period_of_performance_start_date ordering_period_end_date  \
0                       2020-08-15               2025-08-14   
1                       2024-04-17               2029-04-16   
2                       2020-04-01               2030-03-31   
3                       2022-10-07               2027-10-06   
4                       2015-11-03               2025-11-02   

              awarding_agency_name     awarding_sub_agency_name  \
0  General Services Administration  Federal Acquisition Service   
1  General Services Admi

In [ ]:
df_subset.to_csv('filtered_contracts.csv', index=False)

In [ ]:

keep_cols_indices = [
    2, 10, 15, 23, 26, 29, 31, 35, 37, 44, 50, 53, 59, 60, 62, 64, 65, 66, 67, 68, 71,
    109, 110, 144, 159, 161, 163, 176, 187, 186, 187, 194, 195
]

# Add the range from 195 to 296 (inclusive)
keep_cols_indices.extend(range(196, 297))

# Get the column names from the first few rows
temp_df = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', nrows=100)
all_cols = list(temp_df.columns)

# Select the column names based on the indices
keep_cols_names = [all_cols[i] for i in keep_cols_indices if i < len(all_cols)]

# Read the full CSV file, keeping only the specified columns
df_subset2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', usecols=keep_cols_names)

# Display the first few rows of the new dataframe
print(df_subset2.head())

# Display info about the new dataframe
df_subset2.info()
df_subset2.to_csv('filtered_contracts2.csv', index=False)

<ipython-input-33-8ff5d90c4d14>:17: DtypeWarning: Columns (71) have mixed types. Specify dtype option on import or set low_memory=False.
  df_subset2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', usecols=keep_cols_names)


   award_id_piid  total_dollars_obligated  potential_total_value_of_award  \
0  SPE2DM25FP2GX                   211.68                          211.68   
1  47QSWA25P0BWK                  7794.00                         7794.00   
2  SPE7LX25FAR59                    10.35                           10.35   
3  SPE30025FRFK1                  8902.92                         8902.92   
4  SPE8ES25F914R                   173.39                          173.39   

  period_of_performance_start_date ordering_period_end_date  \
0                       2025-01-08                      NaN   
1                       2025-01-08                      NaN   
2                       2025-01-08                      NaN   
3                       2025-01-08                      NaN   
4                       2025-01-08                      NaN   

              awarding_agency_name     awarding_sub_agency_name  \
0            Department of Defense     Defense Logistics Agency   
1  General Services Admi

In [ ]:

keep_cols_indices = [
    2, 10, 15, 23, 26, 29, 31, 35, 37, 44, 50, 53, 59, 60, 62, 64, 65, 66, 67, 68, 71,
    109, 110, 144, 159, 161, 163, 176, 187, 186, 187, 194, 195
]

# Add the range from 195 to 296 (inclusive)
keep_cols_indices.extend(range(196, 297))

# Get the column names from the first few rows
temp_df = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv', nrows=100)
all_cols = list(temp_df.columns)

# Select the column names based on the indices
keep_cols_names = [all_cols[i] for i in keep_cols_indices if i < len(all_cols)]

# Read the full CSV file, keeping only the specified columns
df_subset3 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv', usecols=keep_cols_names)

# Display the first few rows of the new dataframe
print(df_subset3.head())

# Display info about the new dataframe
df_subset3.info()
df_subset3.to_csv('filtered_contracts3.csv', index=False)

<ipython-input-34-ee8ae4957674>:17: DtypeWarning: Columns (71) have mixed types. Specify dtype option on import or set low_memory=False.
  df_subset3 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv', usecols=keep_cols_names)


   award_id_piid  total_dollars_obligated  potential_total_value_of_award  \
0  SPE2DV25FG9P1                  1036.21                         1036.21   
1  36F79725D0010                     0.00                       225000.00   
2  N6523625F0013               2905784.41                      2905784.41   
3  SPE4A625FA7DB                   151.98                          151.98   
4  47QSSC25F11PD                   410.04                          410.04   

  period_of_performance_start_date ordering_period_end_date  \
0                       2024-10-31                      NaN   
1                       2024-11-01               2029-10-31   
2                       2024-10-18                      NaN   
3                       2024-10-31                      NaN   
4                       2024-10-31                      NaN   

              awarding_agency_name        awarding_sub_agency_name  \
0            Department of Defense        Defense Logistics Agency   
1   Department of 

In [ ]:
df_filtered1 = pd.read_csv('/content/filtered_contracts.csv')
df_filtered2 = pd.read_csv('/content/filtered_contracts2.csv')
df_filtered3 = pd.read_csv('/content/filtered_contracts3.csv')

df_combined = pd.concat([df_filtered1, df_filtered2, df_filtered3], ignore_index=True)
df_combined.info()
print(df_combined.head())
df_combined.to_csv('combined_filtered_contracts.csv', index=True)

<ipython-input-5-5f9dd49645d0>:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_filtered1 = pd.read_csv('/content/filtered_contracts.csv')
<ipython-input-5-5f9dd49645d0>:2: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_filtered2 = pd.read_csv('/content/filtered_contracts2.csv')
<ipython-input-5-5f9dd49645d0>:3: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_filtered3 = pd.read_csv('/content/filtered_contracts3.csv')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500726 entries, 0 to 2500725
Columns: 133 entries, award_id_piid to last_modified_date
dtypes: float64(9), object(124)
memory usage: 2.5+ GB
   award_id_piid  total_dollars_obligated  potential_total_value_of_award  \
0  47QSWA20D0092                      0.0                        500000.0   
1  47QRAA24D006P                      0.0                        500000.0   
2  47QTCA20D008C                      0.0                       4000000.0   
3  47QTCA23D0009                      0.0                       7600000.0   
4     GS33F004DA                      0.0                        450000.0   

  period_of_performance_start_date ordering_period_end_date  \
0                       2020-08-15               2025-08-14   
1                       2024-04-17               2029-04-16   
2                       2020-04-01               2030-03-31   
3                       2022-10-07               2027-10-06   
4                       2015-1

In [ ]:
df_combined.shape

(2500726, 133)

In [ ]:
df_combined.head(10).transpose().reset_index()

,index,0,1,2,3,4,5,6,7,8,9
0,award_id_piid,47QSWA20D0092,47QRAA24D006P,47QTCA20D008C,47QTCA23D0009,GS33F004DA,47QTCA19D0063,GS03F060AA,47QTCA23D00EN,47QRAA24D00CT,47QSWA22D0036
1,total_dollars_obligated,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,potential_total_value_of_award,500000.0,500000.0,4000000.0,7600000.0,450000.0,6000000.0,708183.0,500000.0,2000000.0,500000.0
3,period_of_performance_start_date,2020-08-15,2024-04-17,2020-04-01,2022-10-07,2015-11-03,2019-02-19,2013-04-01,2023-09-26,2024-08-22,2022-02-11
4,ordering_period_end_date,2025-08-14,2029-04-16,2030-03-31,2027-10-06,2025-11-02,2029-02-18,2028-03-31,2028-09-25,2029-08-21,2027-02-10
5,awarding_agency_name,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration
6,awarding_sub_agency_name,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service
7,funding_agency_name,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration,General Services Administration
8,funding_sub_agency_name,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service,Federal Acquisition Service
9,foreign_funding,X,X,X,X,X,X,X,X,X,X


In [ ]:
pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 1000)

In [ ]:
df_subset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Columns: 133 entries, award_id_piid to last_modified_date
dtypes: float64(9), object(124)
memory usage: 1014.7+ MB


In [ ]:
df_subset2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Columns: 133 entries, award_id_piid to last_modified_date
dtypes: float64(9), object(124)
memory usage: 1014.7+ MB


In [ ]:
df_subset3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500726 entries, 0 to 500725
Columns: 133 entries, award_id_piid to last_modified_date
dtypes: float64(9), object(124)
memory usage: 508.1+ MB


In [ ]:
df1 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', nrows=500_000)


<ipython-input-2-f6a00191ef57>:1: DtypeWarning: Columns (5,47,68,71,72,81,107,108,115,116,117,118,135,136,165,171,172,179,180) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', nrows=500_000)


In [ ]:
df2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', skiprows=500_000, header=None)


<ipython-input-3-db00d4cad4a3>:1: DtypeWarning: Columns (5,47,68,71,72,81,107,108,115,116,135,136,164,165,179,180) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_1.csv', skiprows=500_000, header=None)


In [ ]:
df1.to_parquet('df1.parquet')
#df2.to_parquet('df2.parquet')

ArrowTypeError: ("Expected bytes, got a 'float' object", 'Conversion failed for column parent_award_agency_id with type object')

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Columns: 297 entries, contract_transaction_unique_key to last_modified_date
dtypes: float64(28), int64(3), object(266)
memory usage: 1.1+ GB


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Columns: 297 entries, 12C2_4732_12318724F0301_P00001_47QTCA20D00B1_0 to 2025-02-11.1
dtypes: float64(30), int64(3), object(264)
memory usage: 1.1+ GB


In [ ]:
df2.tail()

,12C2_4732_12318724F0301_P00001_47QTCA20D00B1_0,CONT_AWD_12318724F0301_12C2_47QTCA20D00B1_4732,12318724F0301,P00001,0,4732,FEDERAL ACQUISITION SERVICE,47QTCA20D00B1,PA0002,100000.00,...,Unnamed: 287,Unnamed: 288,Unnamed: 289,Unnamed: 290,Unnamed: 291,Unnamed: 292,Unnamed: 293,https://www.usaspending.gov/award/CONT_AWD_12318724F0301_12C2_47QTCA20D00B1_4732/,2025-02-10,2025-02-11.1
499995,9700_9700_SPE2D625F17MW_0_SPE2DE24D0021_0,CONT_AWD_SPE2D625F17MW_9700_SPE2DE24D0021_9700,SPE2D625F17MW,0,0.0,9700,DEPT OF DEFENSE,SPE2DE24D0021,0,94.58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.usaspending.gov/award/CONT_AWD_SPE...,2025-01-08,2025-01-16
499996,12C2_12C2_1240BE24F0035_P00002_1204N020A0048_0,CONT_AWD_1240BE24F0035_12C2_1204N020A0048_12C2,1240BE24F0035,P00002,0.0,12C2,FOREST SERVICE,1204N020A0048,0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.usaspending.gov/award/CONT_AWD_124...,2025-01-08,2025-01-08
499997,4732_-NONE-_47QSSC25P0HJV_0_-NONE-_0,CONT_AWD_47QSSC25P0HJV_4732_-NONE-_-NONE-,47QSSC25P0HJV,0,0.0,NaN,NaN,NaN,NaN,80.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.usaspending.gov/award/CONT_AWD_47Q...,2025-01-09,2025-01-09
499998,9700_9700_SPE30025FRF5C_0_SPE30025DV002_0,CONT_AWD_SPE30025FRF5C_9700_SPE30025DV002_9700,SPE30025FRF5C,0,0.0,9700,DEPT OF DEFENSE,SPE30025DV002,0,242.60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.usaspending.gov/award/CONT_AWD_SPE...,2025-01-08,2025-03-21
499999,1540_1540_15B61225F00000020_0_15B61220D00000001_0,CONT_AWD_15B61225F00000020_1540_15B61220D00000...,15B61225F00000020,0,0.0,1540,FEDERAL PRISON SYSTEM / BUREAU OF PRISONS,15B61220D00000001,0,6000.00,...,240000.0,BRYAN HEFTLER,156402.0,BOBBIE J HEFTLER,187722.0,GREGORY J BARBEREE,187000.0,https://www.usaspending.gov/award/CONT_AWD_15B...,2025-01-08,2025-01-08


In [ ]:
df4 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', nrows=500_000)


<ipython-input-13-c8c35ef9c0c5>:1: DtypeWarning: Columns (5,47,71,72,81,115,135,136,164,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df4 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', nrows=500_000)


In [ ]:
df5 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv', skiprows=500_000, header=None)


TypeError: read_csv() got an unexpected keyword argument 'headers'

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Columns: 297 entries, contract_transaction_unique_key to last_modified_date
dtypes: float64(84), int64(6), object(207)
memory usage: 232.2+ KB


In [ ]:
df2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv')




<ipython-input-8-9cc200530812>:1: DtypeWarning: Columns (5,47,71,72,81,115,135,136,164,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_2.csv')


In [ ]:
df3 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv')


<ipython-input-3-ad108dc0422d>:1: DtypeWarning: Columns (5,47,71,72,81,115,135,136,164,165) have mixed types. Specify dtype option on import or set low_memory=False.
  df3 = pd.read_csv('./FY2025_All_Contracts_Full_20250508_3.csv')


In [ ]:
full_contracts = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

full_contracts.drop_duplicates(inplace=True)

print(full_contracts.head())
#full_contracts.info()

**Savings**

In [ ]:
# grants

In [ ]:
#contracts

In [ ]:
#leases

**Payments**

In [ ]:
#payments

In [ ]:
#statistics

GITHUB

In [ ]:
# import data from url and view as data frame

import pandas as pd

# Define the URL of the CSV file
url = 'https://github.com/etmcrae/DOGE-Clone/blob/main/data/doge-contract.csv'

# Read the CSV data from the URL into a pandas DataFrame
df = pd.read_csv(url)

# Display the first few rows of the DataFrame
print(df.head())

ParserError: Error tokenizing data. C error: Expected 1 fields in line 43, saw 66


In [ ]:
import requests

url = 'https://github.com/etmcrae/DOGE-Clone/blob/main/data/doge-contract.csv'
url.split('/')[-1]

print(content)









<!DOCTYPE html>
<html
  lang="en"
  
  data-color-mode="auto" data-light-theme="light" data-dark-theme="dark"
  data-a11y-animated-images="system" data-a11y-link-underlines="true"
  
  >



  <head>
    <meta charset="utf-8">
  <link rel="dns-prefetch" href="https://github.githubassets.com">
  <link rel="dns-prefetch" href="https://avatars.githubusercontent.com">
  <link rel="dns-prefetch" href="https://github-cloud.s3.amazonaws.com">
  <link rel="dns-prefetch" href="https://user-images.githubusercontent.com/">
  <link rel="preconnect" href="https://github.githubassets.com" crossorigin>
  <link rel="preconnect" href="https://avatars.githubusercontent.com">

  


  <link crossorigin="anonymous" media="all" rel="stylesheet" href="https://github.githubassets.com/assets/light-74231a1f3bbb.css" /><link crossorigin="anonymous" media="all" rel="stylesheet" href="https://github.githubassets.com/assets/light_high_contrast-83beb16e0ecf.css" /><link crossorigin="anonymous" media="all" rel="

In [ ]:
combined_filtered_contracts.head()

NameError: name 'combined_filtered_contracts' is not defined

In [ ]:
combined_filtered_contracts.head(10).transpose()

NameError: name 'combined_filtered_contracts' is not defined

In [ ]:
combined_filtered_contracts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500726 entries, 0 to 2500725
Columns: 133 entries, award_id_piid to last_modified_date
dtypes: float64(9), object(124)
memory usage: 2.5+ GB


In [ ]:
filtered_contracts.csv

In [ ]:
combined_filtered_contracts.head

NameError: name 'combined_filtered_contracts' is not defined

In [ ]:
# prompt: curl -X 'GET' \
#   'https://api.doge.gov/savings/contracts?per_page=500' \
#   -H 'accept: application/json'
import requests
url = 'https://api.doge.gov/savings/contracts?per_page=500'
headers = {'accept': 'application/json'}

response = requests.get(url, headers=headers)

if response.status_code == 200:
  data = response.json()
  # You can now work with the JSON data in the 'data' variable
  print("Successfully fetched data from the API.")
  # Example: print the first item in the results list if it exists
  if data and 'results' in data and data['results']:
    print("First result:", data['results'][0])
else:
  print(f"Error fetching data. Status code: {response.status_code}")
  print("Response body:", response.text)

Successfully fetched data from the API.


In [ ]:
# data from api save to a df called DOGE
dfs = []
for i in range (1, 24):

  url = f'https://api.doge.gov/savings/contracts?page={i}&per_page=500'
  headers = {'accept': 'application/json'}

  response = requests.get(url, headers=headers)

  if response.status_code == 200:
    data = response.json()
    # Assuming the data is in a 'results' key in the JSON response
    if data and 'result' in data:
      DOGE = pd.DataFrame(data['result']['contracts'])
      dfs.append(DOGE)
      print("Successfully created DataFrame DOGE.")
      print(DOGE.head())
    else:
      print("API response does not contain 'results' key.")
  else:
    print(f"Error fetching data. Status code: {response.status_code}")
    print("Response body:", response.text)


Successfully created DataFrame DOGE.
             piid                                   agency  \
0  2032H824F00058                   Department of Treasury   
1  20341522F00039                   Department of Treasury   
2  75D30124F19709  Department of Health and Human Services   
3     Unavailable                                    USAID   
4   95170024C0273    United States Agency for Global Media   

                           vendor         value  \
0         DELOITTE CONSULTING LLP  7.259739e+06   
1             FI CONSULTING, INC.  2.992678e+06   
2     FAMILY HEALTH INTERNATIONAL  1.428250e+08   
3                     Unavailable  9.499996e+07   
4  MISCELLANEOUS FOREIGN AWARDEES  1.767699e+07   

                                         description  fpds_status  \
0  Comprehensive Training Strategy aimed at enhan...   TERMINATED   
1            Independent Verification and Validation   TERMINATED   
2  HUMAN IMMUNODEFICIENCY VIRUS COMMUNICATIONS PR...   TERMINATED   
3      

In [ ]:
len(dfs)

23

In [ ]:
dfs[5]

,piid,agency,vendor,value,description,fpds_status,fpds_link,deleted_date,savings
0,75F40124F19009,Department of Health and Human Services,SOLUTION PLANNING AND CONTRACT ENVIRONMENT INC...,47046.63,WHITE PLAINS RP FURNITURE-RELOCATION RECONFIGU...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/24/2025,0.00
1,75F40124F19013,Department of Health and Human Services,VETS SYNERGETIC GROUP LLC,164261.60,SCIENTIFIC SUPPORT FOR STEM CELL CHARACTERIZATION,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/24/2025,0.00
2,75N93020F00001,Department of Health and Human Services,"ADVANCED BIOSCIENCE LABORATORIES, INC.",31202.00,ADMINISTRATIVE ASST,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/24/2025,0.00
3,75N93024F00012,Department of Health and Human Services,CAMRIS INTERNATIONAL INC.,584563.20,"NIAID PROFESSIONAL, SCIENTIFIC, AND TECHNICAL ...",TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/24/2025,394310.40
4,75N93024F00057,Department of Health and Human Services,"GAP SOLUTIONS, INC.",583430.40,"NIAID PROFESSIONAL, SCIENTIFIC, AND TECHNICAL ...",TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/24/2025,393542.40
...,...,...,...,...,...,...,...,...,...
495,Unavailable,USAID,Unavailable,476636.40,Unavailable for legal reasons,Unavailable,https://fpds.gov,4/16/2025,0.00
496,Unavailable,USAID,Unavailable,672508.00,Unavailable for legal reasons,Unavailable,https://fpds.gov,4/16/2025,446222.00
497,75N93023C00012,Department of Health and Human Services,PLURI BIOTECH LTD,4175467.00,DEVELOPMENT OF PLX-R18 CELL THERAPY AS A COUNT...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/16/2025,1313852.00
498,75D30122C14874,Department of Health and Human Services,"KATMAI RESOURCE MANAGEMENT, LLC",24970188.18,OSH PROFESSIONAL/TECHNICAL SUPPORT SERVICES (P...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,4/16/2025,10009256.46


In [ ]:
pd.concat(dfs, ignore_index=True)

,piid,agency,vendor,value,description,fpds_status,fpds_link,deleted_date,savings
0,2032H824F00058,Department of Treasury,DELOITTE CONSULTING LLP,7.259739e+06,Comprehensive Training Strategy aimed at enhan...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,6/3/2025,4.354402e+06
1,20341522F00039,Department of Treasury,"FI CONSULTING, INC.",2.992678e+06,Independent Verification and Validation,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,6/3/2025,0.000000e+00
2,75D30124F19709,Department of Health and Human Services,FAMILY HEALTH INTERNATIONAL,1.428250e+08,HUMAN IMMUNODEFICIENCY VIRUS COMMUNICATIONS PR...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,6/3/2025,1.245576e+08
3,Unavailable,USAID,Unavailable,9.499996e+07,Unavailable for legal reasons,Unavailable,https://fpds.gov,6/3/2025,6.524996e+07
4,95170024C0273,United States Agency for Global Media,MISCELLANEOUS FOREIGN AWARDEES,1.767699e+07,MEDIUM WAVE RADIO BROADCASTING TRANSMISSION SE...,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,6/3/2025,1.608278e+07
...,...,...,...,...,...,...,...,...,...
11037,2032H321A00002,Department of Treasury,Hispanic Assoc. of Colleges and Universities,6.500000e+06,DEIA Training,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,1/22/2025,6.500000e+06
11038,75N96024F00045,Department of Health and Human Services,Powertrain Inc.,2.093948e+05,DEIA Training,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,1/22/2025,0.000000e+00
11039,2032H824F00064,Department of Treasury,"STEEL POINT SOLUTIONS, LLC",5.265565e+06,DEIA Training,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,1/22/2025,2.323823e+06
11040,12314423C0053,Department of Agriculture,"TIMOTHY J. LONDAGIN, LLC",2.035756e+06,DEIA Training,TERMINATED,https://www.fpds.gov/ezsearch/jsp/viewLinkCont...,1/22/2025,9.202590e+05


In [ ]:
contracts = pd.concat(dfs, ignore_index=True)

In [ ]:
# save concat to google drive as contracts

from google.colab import drive
drive.mount('/content/drive')
contracts.to_csv('/content/drive/My Drive/DOGE/contracts.csv', index=False)

print("DataFrame 'contracts' saved to Google Drive as contracts.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DataFrame 'full_contracts' saved to Google Drive as contracts.csv


In [ ]:
url = 'https://api.doge.gov/savings/contracts'
headers = {'accept': 'application/json'}

response = requests.get(url, headers=headers)

if response.status_code == 200:
  data = response.json()

  if data and 'result' in data:
    DOGE = pd.DataFrame(data['result'])
    print("Successfully created DataFrame DOGE.")
    print(DOGE.head())
  else:
    print("API response does not contain 'result' key.")
else:
  print(f"Error fetching data. Status code: {response.status_code}")
  print("Response body:", response.text)

Successfully created DataFrame DOGE.
                                           contracts
0  {'piid': '2032H824F00058', 'agency': 'Departme...
1  {'piid': '20341522F00039', 'agency': 'Departme...
2  {'piid': '75D30124F19709', 'agency': 'Departme...
3  {'piid': 'Unavailable', 'agency': 'USAID', 've...
4  {'piid': '95170024C0273', 'agency': 'United St...


In [ ]:
# prompt: print DOGE dataframe info

DOGE.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   contracts  100 non-null    object
dtypes: object(1)
memory usage: 932.0+ bytes
